# 03 — Severity Classification

Fine-tune Bio_ClinicalBERT on MTSamples for severity classification.

**What this notebook covers**
- Inspect weak-supervision labels
- Train the classifier
- Evaluate on the test set
- Save metrics for the dashboard
- Run inference on new notes

> **Note**: Training takes ~20 min on CPU, ~5 min on Colab GPU.

## 1. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import plotly.express as px
from pathlib import Path

from src.etl.extract import load_mtsamples
from src.etl.transform import prepare_for_training
from src.nlp.classifier import ClinicalClassifier
from src.utils.config import ModelConfig

print('Setup OK')

Setup OK


## 2. Load and Inspect Training Data

In [2]:
raw_df = load_mtsamples()
df     = prepare_for_training(raw_df)

print(f'Training samples: {len(df):,}')
print()
print('Severity distribution:')
print(df['severity'].value_counts())
print()
print(f'Class balance:')
for label, count in df['severity'].value_counts().items():
    print(f'  {label:<12} {count:>5}  ({count/len(df):.1%})')

2026-06-26 19:15:41 | INFO     | src.etl.extract                | Loading MTSamples from C:\Users\Hp\Documents\clinical-nlp-pipeline\data\raw\mtsamples.csv
2026-06-26 19:15:42 | INFO     | src.etl.extract                | MTSamples loaded: 4999 notes, 40 specialties
2026-06-26 19:15:42 | INFO     | src.etl.transform              | Running full transformation pipeline...
2026-06-26 19:15:42 | INFO     | src.etl.transform              | Dropped 33 rows with missing transcription text
2026-06-26 19:16:11 | INFO     | src.etl.transform              | Removed 2609 duplicate notes
2026-06-26 19:16:11 | INFO     | src.etl.transform              | Clean notes: 2357 rows remaining
2026-06-26 19:16:11 | INFO     | src.etl.transform              | Filtered 33 notes shorter than 30 words
2026-06-26 19:16:11 | INFO     | src.etl.transform              | Deriving severity labels via weak supervision...
2026-06-26 19:16:20 | INFO     | src.etl.transform              |   urgent     1053  (45.3%)
2026-

In [3]:
# Distribution chart
fig = px.pie(
    df, names='severity',
    title='Weak-supervision severity label distribution',
    color='severity',
    color_discrete_map={'routine': '#2ecc71', 'urgent': '#f39c12', 'critical': '#e74c3c'},
    hole=0.35,
)
fig.show()

## 3. Train the Classifier

This cell fine-tunes Bio_ClinicalBERT and saves the best checkpoint.

In [5]:
clf     = ClinicalClassifier(task='severity')
metrics = clf.train(df)

print()
print('Training complete!')
print(f'  Val  accuracy : {metrics["val_accuracy"]:.4f}')
print(f'  Val  F1       : {metrics["val_f1"]:.4f}')
print(f'  Test accuracy : {metrics["test_accuracy"]:.4f}')
print(f'  Test F1       : {metrics["test_f1"]:.4f}')

2026-06-23 15:33:53 | INFO     | src.nlp.classifier             | ClinicalClassifier init: task=severity, model=emilyalsentzer/Bio_ClinicalBERT, labels=['routine', 'urgent', 'critical']
2026-06-23 15:33:53 | INFO     | src.nlp.classifier             | Starting fine-tuning: task=severity
2026-06-23 15:35:39 | INFO     | src.nlp.classifier             | Loading tokenizer: emilyalsentzer/Bio_ClinicalBERT
2026-06-23 15:35:40 | INFO     | src.nlp.classifier             | Training data: 2324 notes, label dist={'routine': 875, 'urgent': 1098, 'critical': 351}
2026-06-23 15:35:40 | INFO     | src.nlp.classifier             | Split: train=1742  val=349  test=233


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

2026-06-23 15:35:44 | INFO     | src.nlp.classifier             | Training on device: cpu
2026-06-23 15:35:45 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT "HTTP/1.1 200 OK"
2026-06-23 15:35:46 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/commits/main "HTTP/1.1 200 OK"
2026-06-23 15:35:46 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/discussions?p=0 "HTTP/1.1 200 OK"
2026-06-23 15:35:47 | INFO     | httpx                          | HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/commits/refs%2Fpr%2F16 "HTTP/1.1 200 OK"
2026-06-23 15:35:47 | INFO     | httpx                          | HTTP Request: HEAD https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT/resolve/refs%2Fpr%2F16/model.safetensors.index

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

2026-06-25 09:57:14 | INFO     | src.nlp.classifier             | Model saved to C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier
2026-06-25 09:57:14 | INFO     | src.nlp.classifier             |   → New best model saved (val_f1=0.7144)
2026-06-25 13:15:42 | INFO     | src.nlp.classifier             | Epoch 3/3  loss=0.4488  val_acc=0.7249  val_f1=0.7243


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-25 13:15:46 | INFO     | src.nlp.classifier             | Model saved to C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier
2026-06-25 13:15:46 | INFO     | src.nlp.classifier             |   → New best model saved (val_f1=0.7243)
2026-06-25 13:15:46 | INFO     | src.nlp.classifier             | Loading best checkpoint for test evaluation...
2026-06-25 13:15:46 | INFO     | src.nlp.classifier             | Loading classifier from: C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-06-25 13:15:48 | INFO     | src.nlp.classifier             | Classifier loaded ✓
2026-06-25 13:20:23 | INFO     | src.nlp.classifier             | Training complete — test_acc=0.7210  test_f1=0.7193

Training complete!
  Val  accuracy : 0.7249
  Val  F1       : 0.7243
  Test accuracy : 0.7210
  Test F1       : 0.7193


## 3b. Per-Class Performance (Test Set)

Weighted accuracy/F1 can hide poor performance on the minority class.
"critical" is only ~15% of the data but the most clinically costly class
to get wrong (a missed critical case is worse than an over-flagged routine one).
This reconstructs the exact held-out test split used during training
(same seed, same filtering) and checks per-class precision/recall.

In [5]:
clf = ClinicalClassifier(task='severity')


2026-06-26 19:20:28 | INFO     | src.nlp.classifier             | ClinicalClassifier init: task=severity, model=emilyalsentzer/Bio_ClinicalBERT, labels=['routine', 'urgent', 'critical']


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px

from src.utils.config import ClassifierConfig, TrainingConfig

labels_list = ClassifierConfig.active_labels()  # ['routine', 'urgent', 'critical']
label2id    = {lbl: i for i, lbl in enumerate(labels_list)}
cfg         = TrainingConfig

# Reconstruct the exact same split used inside clf.train() — same filtering,
# same integer label encoding, same seed — so this evaluates on the untouched
# held-out test set rather than data the model has already seen.
valid_mask = df['severity'].isin(labels_list)
df_valid   = df[valid_mask]
texts      = df_valid['transcription'].tolist()
label_ids  = [label2id[lbl] for lbl in df_valid['severity']]

x_trainval, x_test, y_trainval, y_test_ids = train_test_split(
    texts, label_ids,
    test_size    = cfg.test_split,
    random_state = cfg.random_seed,
    stratify     = label_ids,
)
y_test = [labels_list[i] for i in y_test_ids]

# Load the best saved checkpoint and predict on the held-out test set
clf.load()
results = clf.predict_batch(x_test)
y_pred  = [r.label for r in results]

print(classification_report(y_test, y_pred, labels=labels_list, digits=3))

cm = confusion_matrix(y_test, y_pred, labels=labels_list)
fig = px.imshow(
    cm, text_auto=True,
    x=labels_list, y=labels_list,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    title='Confusion Matrix — Severity Classification (Test Set)',
    color_continuous_scale='Blues',
)
fig.show()

2026-06-26 19:21:50 | INFO     | src.nlp.classifier             | Loading classifier from: C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-06-26 19:21:52 | INFO     | src.nlp.classifier             | Loading tokenizer: emilyalsentzer/Bio_ClinicalBERT
2026-06-26 19:21:53 | INFO     | src.nlp.classifier             | Classifier loaded OK
              precision    recall  f1-score   support

     routine      0.865     0.837     0.851        92
      urgent      0.826     0.849     0.837       106
    critical      0.800     0.800     0.800        35

    accuracy                          0.837       233
   macro avg      0.830     0.829     0.829       233
weighted avg      0.837     0.837     0.837       233



In [9]:
print(len(x_test), len(y_test), len(y_pred))
print(pd.Series(y_test).value_counts())


233 233 233
urgent      110
routine      88
critical     35
Name: count, dtype: int64


## 4. Save Metrics for the Dashboard

In [10]:
metrics_path = ModelConfig.fine_tuned_dir / 'training_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)

with open(metrics_path, 'w') as fh:
    json.dump(metrics, fh, indent=2)

print(f'Metrics saved to {metrics_path}')

Metrics saved to C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier\training_metrics.json


## 5. Inference on New Notes

In [12]:
clf.load()  # load the best saved checkpoint

test_notes = [
    'Patient presents for routine annual check-up. No acute complaints. Vitals stable.',
    'Admitted to ED with acute severe abdominal pain, fever 39.5C, elevated WBC.',
    'Patient transferred to ICU. Intubated for respiratory failure. GCS 6.',
]

for note in test_notes:
    result = clf.predict(note)
    print(f'  [{result.label.upper():<8}] {result.confidence:.0%}  {note[:70]}...')

2026-06-25 15:59:39 | INFO     | src.nlp.classifier             | Loading classifier from: C:\Users\Hp\Documents\clinical-nlp-pipeline\data\models\severity_classifier


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-06-25 15:59:39 | INFO     | src.nlp.classifier             | Classifier loaded ✓
  [URGENT  ] 98%  Patient presents for routine annual check-up. No acute complaints. Vit...
  [URGENT  ] 50%  Admitted to ED with acute severe abdominal pain, fever 39.5C, elevated...
  [CRITICAL] 89%  Patient transferred to ICU. Intubated for respiratory failure. GCS 6....
